# Hệ thống xếp thời khóa biểu bằng Genetic Algorithm trên Google Colab

Đây là bản notebook **tự chứa** dành cho chạy và demo. Source code, dataset EASY/MEDIUM và các kết quả mẫu cần cho giao diện đã được nén ngay trong notebook; không cần upload thêm thư mục hoặc file ZIP.

> Thuật toán chủ yếu sử dụng CPU nên không bắt buộc bật GPU. Dữ liệu trong `/content` sẽ mất khi phiên Colab kết thúc; hãy tải kết quả về máy.

## 1. Chuẩn bị dự án tự động

Chỉ cần chạy cell bên dưới. Cell sẽ giải nén bản source tích hợp vào `/content/genetic-alo-main`; hoàn toàn không hiện hộp thoại yêu cầu upload.

In [ ]:
#@title Chuẩn bị source code tự động (không cần upload file ZIP)
# Source code và dataset đã được nén ngay trong notebook này.
import base64
import hashlib
import io
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

DU_LIEU_SOURCE_BASE64 = (
    "..."
    "YXBwLnB5UEsFBgAAAAA3ADcAzw8AAMYbCQAAAA=="
)
MA_BAM_SOURCE = "92df18de6f31dc27432c3931f3ecfe6f28b24623d2f5d022bf28c3e855f8dbd0"

# Giải mã gói source đang được lưu bên trong notebook.
du_lieu_zip = base64.b64decode(DU_LIEU_SOURCE_BASE64)
if hashlib.sha256(du_lieu_zip).hexdigest() != MA_BAM_SOURCE:
    raise ValueError("Gói source tích hợp bị sai mã kiểm tra; hãy tải lại notebook.")

THU_MUC_DU_AN = Path("/content/genetic-alo-main")
# Làm sạch bản chạy cũ để có thể bấm Run all nhiều lần trong cùng một phiên.
if THU_MUC_DU_AN.exists():
    shutil.rmtree(THU_MUC_DU_AN)
THU_MUC_DU_AN.mkdir(parents=True)

with zipfile.ZipFile(io.BytesIO(du_lieu_zip), "r") as goi_source:
    goi_source.extractall(THU_MUC_DU_AN)

so_tep = sum(1 for path in THU_MUC_DU_AN.rglob("*") if path.is_file())
print(f"✅ Đã tự động chuẩn bị {so_tep} file tại: {THU_MUC_DU_AN}")
print("Bạn không cần upload thêm source code hoặc dataset.")


## 2. Cài thư viện và kiểm tra cấu trúc dự án

Cell tiếp theo cài đúng các thư viện được khai báo trong `requirements.txt`, chuyển thư mục làm việc về gốc dự án và kiểm tra các file đầu vào quan trọng.

In [ ]:
#@title Cài đặt các thư viện Python
# Dùng chính trình thông dịch Python của Colab để tránh cài nhầm môi trường.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(THU_MUC_DU_AN / "requirements.txt")],
    check=True,
)

# Các import nội bộ như `from dataset import ...` cần chạy từ thư mục gốc dự án.
os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

print("✅ Đã cài đặt thư viện và thiết lập thư mục làm việc.")
print(f"Python: {sys.version.split()[0]}")
print(f"Thư mục hiện tại: {Path.cwd()}")

In [ ]:
#@title Kiểm tra nhanh source code và dữ liệu đầu vào
cac_tep_bat_buoc = [
    THU_MUC_DU_AN / "main.py",
    THU_MUC_DU_AN / "ui_app.py",
    THU_MUC_DU_AN / "requirements.txt",
    THU_MUC_DU_AN / "data/instances/instance_easy.xlsx",
    THU_MUC_DU_AN / "data/instances/instance_medium.xlsx",
]

tep_thieu = [str(tep) for tep in cac_tep_bat_buoc if not tep.is_file()]
if tep_thieu:
    raise FileNotFoundError("Thiếu các file bắt buộc:\n- " + "\n- ".join(tep_thieu))

# Thử import các mô-đun chính để phát hiện sớm lỗi thư viện hoặc đường dẫn.
from dataset import ExcelDatasetLoader
from constraints import ConstraintEvaluator
from ga import GeneticAlgorithmEngine

print("✅ Source code, dataset và các mô-đun chính đều sẵn sàng.")

## 3. Chạy kiểm thử tự động (không bắt buộc)

Bật tùy chọn bên dưới nếu muốn chứng minh source code vượt qua bộ kiểm thử trước khi chạy thuật toán. Bộ test đầy đủ có thể mất vài phút.

In [ ]:
#@title Chạy pytest
CHAY_KIEM_THU = False #@param {type:"boolean"}

if CHAY_KIEM_THU:
    # `check=True` giúp notebook dừng và báo rõ nếu có test thất bại.
    subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=THU_MUC_DU_AN, check=True)
    print("✅ Toàn bộ kiểm thử đã hoàn thành.")
else:
    print("ℹ️ Đã bỏ qua pytest. Có thể bật CHAY_KIEM_THU rồi chạy lại cell này.")

## 4. Chạy thuật toán xếp thời khóa biểu

Cấu hình mặc định chạy bộ dữ liệu `easy`, seed `42`, quần thể `60` và ngân sách `1000` lượt đánh giá. Bật `DUNG_SOFT_LOCAL_SEARCH` để chạy phiên bản đầy đủ **GA + Repair + SLS**.

In [ ]:
#@title Cấu hình và chạy chương trình chính
TEN_DATASET = "easy" #@param ["easy", "medium"]
SEED = 42 #@param {type:"integer"}
KICH_THUOC_QUAN_THE = 60 #@param {type:"integer"}
NGAN_SACH_DANH_GIA = 1000 #@param {type:"integer"}
DUNG_SOFT_LOCAL_SEARCH = True #@param {type:"boolean"}
HO_SO_TRONG_SO = "balanced" #@param ["student_centric", "balanced", "resource_centric"]

duong_dan_input = THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"
thu_muc_ket_qua = THU_MUC_DU_AN / "outputs/colab"
thu_muc_ket_qua.mkdir(parents=True, exist_ok=True)
duong_dan_output = thu_muc_ket_qua / f"best_timetable_{TEN_DATASET}.xlsx"

# Tạo câu lệnh bằng danh sách tham số để xử lý an toàn cả đường dẫn có khoảng trắng.
lenh_chay = [
    sys.executable,
    "main.py",
    "--input", str(duong_dan_input),
    "--output", str(duong_dan_output),
    "--seed", str(SEED),
    "--population-size", str(KICH_THUOC_QUAN_THE),
    "--search-evaluation-budget", str(NGAN_SACH_DANH_GIA),
    "--weight-profile", HO_SO_TRONG_SO,
]

if DUNG_SOFT_LOCAL_SEARCH:
    lenh_chay.append("--soft-local-search")

print("Bắt đầu chạy thuật toán...")
print("Lệnh thực thi:", " ".join(lenh_chay))
subprocess.run(lenh_chay, cwd=THU_MUC_DU_AN, check=True)
print(f"✅ Hoàn thành. File thời khóa biểu: {duong_dan_output}")

## 5. Xem trước và tải kết quả

Cell đầu tiên hiển thị danh sách sheet, một phần bảng thời khóa biểu và biểu đồ hội tụ. Cell kế tiếp nén toàn bộ kết quả của lần chạy Colab để tải về máy.

In [ ]:
#@title Xem trước file Excel và biểu đồ hội tụ
import pandas as pd
from IPython.display import Image, display
from openpyxl import load_workbook

if not duong_dan_output.is_file():
    raise FileNotFoundError("Chưa có file kết quả. Hãy chạy phần 4 trước.")

# Đọc ở chế độ chỉ đọc để xem tên các sheet mà không thay đổi workbook.
workbook = load_workbook(duong_dan_output, read_only=True, data_only=True)
print("Các sheet trong file kết quả:", workbook.sheetnames)
ten_sheet_dau = workbook.sheetnames[0]
workbook.close()

print(f"\nXem trước sheet: {ten_sheet_dau}")
bang_xem_truoc = pd.read_excel(duong_dan_output, sheet_name=ten_sheet_dau)
display(bang_xem_truoc.head(15))

# main.py đặt tên biểu đồ theo phương pháp đã chạy.
ma_phuong_phap = "ga_repair_sls" if DUNG_SOFT_LOCAL_SEARCH else "ga_repair"
duong_dan_bieu_do = thu_muc_ket_qua / f"convergence_{ma_phuong_phap}.png"
if duong_dan_bieu_do.is_file():
    print("Biểu đồ hội tụ:")
    display(Image(filename=str(duong_dan_bieu_do)))
else:
    print("Không tìm thấy biểu đồ hội tụ, nhưng file Excel đã được tạo thành công.")

In [ ]:
#@title Nén và tải toàn bộ kết quả về máy
from google.colab import files

tep_zip_ket_qua = Path("/content/ket_qua_xep_thoi_khoa_bieu")
# make_archive tự thêm phần mở rộng .zip vào tên file.
duong_dan_zip = shutil.make_archive(
    str(tep_zip_ket_qua),
    "zip",
    root_dir=thu_muc_ket_qua,
)
print(f"✅ Đã đóng gói kết quả: {duong_dan_zip}")
files.download(duong_dan_zip)

## 6. Chạy benchmark so sánh phương pháp (tùy chọn)

Benchmark chạy nhiều lần nên lâu hơn chương trình chính. Cấu hình mặc định bên dưới chỉ dùng 3 seed và chế độ `fast` để phù hợp với buổi demo.

In [ ]:
#@title Chạy benchmark
CHAY_BENCHMARK = False #@param {type:"boolean"}
CAC_PHUONG_PHAP = "ga_repair_sls,ga_repair,ga" #@param {type:"string"}
CAC_SEED = "0-2" #@param {type:"string"}

if CHAY_BENCHMARK:
    lenh_benchmark = [
        sys.executable, "main_benchmark.py",
        "--mode", "fast",
        "--methods", CAC_PHUONG_PHAP,
        "--seeds", CAC_SEED,
        "--data-source", "excel",
        "--input", str(duong_dan_input),
        "--experiment-name", "colab_demo",
    ]
    subprocess.run(lenh_benchmark, cwd=THU_MUC_DU_AN, check=True)
    print("✅ Benchmark đã hoàn thành. Kết quả nằm trong outputs/benchmark/colab_demo.")
else:
    print("ℹ️ Đã bỏ qua benchmark. Bật CHAY_BENCHMARK nếu cần chạy so sánh.")

## 7. Mở giao diện Streamlit trên Colab (tùy chọn)

Colab không công khai trực tiếp cổng `8501`, nên cell dưới đây tạo một **Cloudflare Quick Tunnel** tạm thời. Đường link chỉ tồn tại trong phiên Colab hiện tại và phù hợp cho demo/thử nghiệm, không phải triển khai production.

Sau khi URL xuất hiện, mở URL đó trong tab mới. Giữ phiên Colab hoạt động trong lúc sử dụng giao diện.

In [ ]:
#@title Khởi động Streamlit và tạo đường link công khai tạm thời
MO_GIAO_DIEN_STREAMLIT = False #@param {type:"boolean"}

import re
import time

if MO_GIAO_DIEN_STREAMLIT:
    # Nếu cell được chạy lại, dừng các tiến trình cũ do chính notebook đã tạo.
    for ten_bien in ("tien_trinh_streamlit", "tien_trinh_tunnel"):
        tien_trinh_cu = globals().get(ten_bien)
        if tien_trinh_cu is not None and tien_trinh_cu.poll() is None:
            tien_trinh_cu.terminate()

    tep_log_streamlit = Path("/content/streamlit_colab.log")
    tep_log_tunnel = Path("/content/cloudflare_tunnel.log")
    luong_log_streamlit = open(tep_log_streamlit, "w", encoding="utf-8")
    luong_log_tunnel = open(tep_log_tunnel, "w", encoding="utf-8")

    # Chạy Streamlit ở chế độ nền trên cổng 8501.
    tien_trinh_streamlit = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "ui_app.py",
            "--server.headless=true",
            "--server.address=0.0.0.0",
            "--server.port=8501",
        ],
        cwd=THU_MUC_DU_AN,
        stdout=luong_log_streamlit,
        stderr=subprocess.STDOUT,
        text=True,
    )
    time.sleep(8)

    if tien_trinh_streamlit.poll() is not None:
        luong_log_streamlit.flush()
        raise RuntimeError("Streamlit không khởi động được:\n" + tep_log_streamlit.read_text(errors="replace"))

    # Wrangler tạo Quick Tunnel miễn phí, không yêu cầu tài khoản hay token.
    tien_trinh_tunnel = subprocess.Popen(
        ["npx", "--yes", "wrangler", "tunnel", "quick-start", "http://localhost:8501"],
        stdout=luong_log_tunnel,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Chờ tối đa 90 giây để tunnel cấp một URL trycloudflare.com.
    url_cong_khai = None
    for _ in range(90):
        time.sleep(1)
        luong_log_tunnel.flush()
        noi_dung_log = tep_log_tunnel.read_text(encoding="utf-8", errors="replace")
        ket_qua_url = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", noi_dung_log)
        if ket_qua_url:
            url_cong_khai = ket_qua_url.group(0)
            break
        if tien_trinh_tunnel.poll() is not None:
            break

    if url_cong_khai:
        print("✅ Giao diện Streamlit đang chạy tại:")
        print(url_cong_khai)
        print("Hãy giữ phiên Colab này hoạt động trong lúc demo.")
    else:
        print(tep_log_tunnel.read_text(encoding="utf-8", errors="replace"))
        raise RuntimeError("Không tạo được URL công khai. Hãy chạy lại cell hoặc kiểm tra kết nối mạng.")
else:
    print("ℹ️ Chưa mở Streamlit. Bật MO_GIAO_DIEN_STREAMLIT rồi chạy lại cell khi cần demo giao diện.")

## Gợi ý trình bày với giảng viên

1. Chạy phần kiểm tra source để chứng minh dữ liệu và mô-đun đã được nạp đúng.
2. Giữ nguyên seed khi cần tái lập cùng một thí nghiệm.
3. Chạy phần thuật toán chính và trình bày log số vi phạm cứng, điểm phạt mềm cùng file Excel đầu ra.
4. Chỉ chạy benchmark khi cần so sánh phương pháp vì phần này tốn nhiều thời gian hơn.
5. Luôn tải file ZIP kết quả trước khi đóng phiên Colab.